<a href="https://colab.research.google.com/github/raaghavkk/UG04-NLP-COMM061/blob/main/notebooks/deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio transformers torch

In [4]:
import gradio as gr
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Load tokenizer once
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Model registry
MODEL_REGISTRY = {
    "Australian English (en-AU)": "raaghavkk/roberta-sarcasm-en-AU-seed42",
    "Indian English (en-IN)": "raaghavkk/roberta-sarcasm-en-IN-seed42",
    "British English (en-UK)": "raaghavkk/roberta-sarcasm-en-UK-seed42",
}

# Cache loaded models
loaded_models = {}

def get_model(variety):
    if variety not in loaded_models:
        print(f"Loading model for {variety}...")
        loaded_models[variety] = RobertaForSequenceClassification.from_pretrained(MODEL_REGISTRY[variety])
        loaded_models[variety].eval()
    return loaded_models[variety]

def predict(text, variety):
    if not text.strip():
        return "Please enter some text."

    model = get_model(variety)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()
    confidence = torch.softmax(outputs.logits, dim=1).max().item()

    label = "Sarcastic" if prediction == 1 else "Not Sarcastic"
    return f"{label} (confidence: {confidence:.2%})"

# Build Gradio interface
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Textbox(label="Input Text", placeholder="Enter a sentence to analyse..."),
        gr.Dropdown(
            choices=list(MODEL_REGISTRY.keys()),
            label="English Variety",
            value="Australian English (en-AU)"
        )
    ],
    outputs=gr.Textbox(label="Prediction"),
    title="BESSTIE Sarcasm Detector",
    description="Detects sarcasm in Australian, Indian, and British English using variety-specific RoBERTa models.",
    examples=[
        ["Absolute legend, parked his ute right across my driveway. Good onya mate.", "Australian English (en-AU)"],
        ["Coz we all have free internet.", "Indian English (en-IN)"],
        ["Oh great, another Monday. Just what I needed.", "British English (en-UK)"],
    ]
)

loaded_models = {}
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc76699e5d65ece931.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import time

test_inputs = [
    "short text",
    "This is a medium length sentence that contains a bit more content than the short one.",
    "This is a much longer piece of text that goes on for quite a while and contains many more words than the previous examples, simulating what a longer review or Reddit comment might look like in the real dataset we used for training."
]

variety = "Australian English (en-AU)"
model = get_model(variety)

print("=== Inference Time Analysis ===\n")
for text in test_inputs:
    times = []
    for _ in range(10):
        start = time.time()
        predict(text, variety)
        end = time.time()
        times.append((end - start) * 1000)
    avg = sum(times) / len(times)
    print(f"Input length: {len(text.split())} words")
    print(f"Average inference time: {avg:.2f}ms\n")

=== Inference Time Analysis ===

Input length: 2 words
Average inference time: 83.29ms

Input length: 16 words
Average inference time: 108.98ms

Input length: 43 words
Average inference time: 180.45ms

